In [2]:
import pickle
import re
import sys
from pathlib import Path

import cairosvg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scanpy as sc

sys.path.insert(0, str(Path.cwd().parent))
from viz_style import apply_style
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.PerSamplePathwayAnalysis.pathway_convergence import gene_sig_at_q
from MixedEffectsModeling.SignalTrendAnalysis.sankey_helpers import strip_reactome_code
from MixedEffectsModeling.SignalTrendAnalysis.run_leading_edge_pattern import CURDIR, SLUG_MAP, Z_THRESH, parse_curation
    
PCDIR = config.PATHWAY_CONV_DIR  # sig.pkl/universe.pkl live in PerSamplePathwayAnalysis, not here
FIGDIR = config.SIGNAL_TREND_FIG_DIR
FIGDIR.mkdir(parents=True, exist_ok=True)

heat_data = pickle.load(open(CURDIR / 'leading_edge_pattern_data.pkl', 'rb'))
summary = pd.read_csv(CURDIR / 'leading_edge_pattern_summary.csv')

In [6]:
DEFAULT_PATIENT_COLOR = '#737373'
MIN_ALPHA = 0.15
MAX_ALPHA = 0.80
SEVERITY_COL = 'Stage/Condition'
SEVERITY_COLORS = {'PDAC': '#d55e00', 'IPMN': '#0072b2', 'Islet Cell Tumor': '#009e73'}
DISPLAY_NAME = {
    'Tuberculosis': 'Tuberculosis', 'Pancreatitis': 'Pancreatitis', 'Pancreatic_Cancer': 'Pancreatic Cancer',
    'Pre-eclampsia': 'Pre-eclampsia', 'Colorectal_Cancer': 'Colorectal Cancer', 'Lung_Cancer': 'Lung Cancer',
    'Esophagus_Cancer': 'Esophagus Cancer', 'Stomach_Cancer': 'Stomach Cancer',
    'Liver_Cancer_Roskams-Hieter': 'Liver Cancer (Roskams-Hieter)', 'Liver_Cancer_Chen': 'Liver Cancer (Chen)',
}

MAX_GENES_PER_PATHWAY = 12
HISTONE_FAMILY_CAP = 2
histone_rx = re.compile(r'^(H1|H2A|H2B|H3|H4)')


def gene_family(sym):
    m = histone_rx.match(sym)
    return m.group(1) if m else sym


def hex_to_rgba(hex_str, a):
    r, g, b = (int(hex_str.lstrip('#')[k:k + 2], 16) for k in (0, 2, 4))
    return f'rgba({r},{g},{b},{a})'


def build_sankey(stem):
    gsea_file, terms = parse_curation(CURDIR / f'{stem}.md')
    slug = SLUG_MAP[stem]
    pdir = PCDIR / slug
    d = pickle.load(open(pdir / 'sig.pkl', 'rb'))
    names_c = d['names_c']
    disp_name = DISPLAY_NAME[stem]
    Zu, Fm = pickle.load(open(pdir / 'universe.pkl', 'rb'))
    sym2idx = {s: i for i, s in enumerate(d['universe_syms'])}
    # [수정] 기존 BH FDR 유의성 계산(bh_sig) 제거

    if stem == 'Pancreatic_Cancer':
        severity = sc.read_h5ad(config.H5AD_PATH, backed='r').obs[SEVERITY_COL].reindex(names_c).values
        pat_color = lambda i: SEVERITY_COLORS.get(severity[i], DEFAULT_PATIENT_COLOR)
        pat_label = lambda i: f'{disp_name} {i + 1}' + (f' ({severity[i]})' if pd.notna(severity[i]) else '')
    else:
        pat_color = lambda i: DEFAULT_PATIENT_COLOR
        pat_label = lambda i: f'{disp_name} {i + 1}'

    cmap = plt.get_cmap('tab20')
    PATH_COLORS = [f'#{"".join(f"{int(c * 255):02x}" for c in cmap(ti % 20)[:3])}' for ti in range(len(terms))]

    pathway_gene_edges = {ti: {} for ti in range(len(terms))}
    for ti, term in enumerate(terms):
        key = (stem, term)
        if key not in heat_data:
            continue
        Zsub, genes = heat_data[key]
        hit = np.abs(Zsub) > Z_THRESH
        if not hit.any():
            continue
        strength = np.where(hit, np.abs(Zsub), 0).sum(axis=0)
        deg = hit.sum(axis=0)
        order = np.lexsort((-strength, -deg))
        picked, family_count = [], {}
        for gc in order:
            if strength[gc] == 0:
                continue
            fam = gene_family(genes[gc])
            if family_count.get(fam, 0) >= HISTONE_FAMILY_CAP:
                continue
            picked.append(gc)
            family_count[fam] = family_count.get(fam, 0) + 1
            if len(picked) == MAX_GENES_PER_PATHWAY:
                break
        for gc in picked:
            edges = {i: Zsub[i, gc] for i in range(len(names_c)) if hit[i, gc]}
            if edges:
                pathway_gene_edges[ti][genes[gc]] = edges

    gene_strength = {}
    for ti, edges_by_gene in pathway_gene_edges.items():
        for gname, edges in edges_by_gene.items():
            gene_strength.setdefault(gname, {})[ti] = sum(abs(z) for z in edges.values())
    if not gene_strength:
        print(f'{stem}: no gene edges survived Z_THRESH, skipping')
        return

    gene_primary_ti = {g: max(dd, key=dd.get) for g, dd in gene_strength.items()}
    displayed_genes = sorted(gene_strength, key=lambda g: (gene_primary_ti[g], -gene_strength[g][gene_primary_ti[g]]))

    pat_gene_z = {}
    for edges_by_gene in pathway_gene_edges.values():
        for gname, edges in edges_by_gene.items():
            for i, z in edges.items():
                pat_gene_z[(i, gname)] = abs(z)

    gene_col = {g: k for k, g in enumerate(displayed_genes)}
    pat_gene_weight = {}
    for (i, gname), z in pat_gene_z.items():
        pat_gene_weight.setdefault(i, {})[gname] = z
    pat_sort_key = {}
    for i, gw in pat_gene_weight.items():
        primary_g = max(gw, key=gw.get)
        pat_sort_key[i] = (gene_col[primary_g], -gw[primary_g])
    displayed_pats = sorted(pat_sort_key, key=pat_sort_key.get)
    used_ti = sorted({ti for ti, e in pathway_gene_edges.items() if e})

    pat_pos = {i: k for k, i in enumerate(displayed_pats)}
    gene_pos = {g: k for k, g in enumerate(displayed_genes)}
    path_pos = {ti: k for k, ti in enumerate(used_ti)}
    n_pat_d, n_gene_d = len(displayed_pats), len(displayed_genes)
    n_pat_total = len(names_c)

    node_labels = ([pat_label(i) for i in displayed_pats] + displayed_genes
                  + [strip_reactome_code(terms[ti]) for ti in used_ti])
    node_colors = ([pat_color(i) for i in displayed_pats] + [PATH_COLORS[gene_primary_ti[g]] for g in displayed_genes]
                  + [PATH_COLORS[ti] for ti in used_ti])

    # [수정] 데이터 내 Z 점수의 최소값과 최대값 계산
    z_vals = list(pat_gene_z.values())
    z_min, z_max = (min(z_vals), max(z_vals)) if z_vals else (Z_THRESH, Z_THRESH + 1.0)

    src, tgt, val, link_col = [], [], [], []
    for (i, gname), z in pat_gene_z.items():
        src.append(pat_pos[i])
        tgt.append(n_pat_d + gene_pos[gname])
        val.append(z)
        
        # [수정] Z 점수에 비례하여 MIN_ALPHA ~ MAX_ALPHA 구간 선형 보간
        alpha = MIN_ALPHA + (MAX_ALPHA - MIN_ALPHA) * ((z - z_min) / (z_max - z_min)) if z_max > z_min else MAX_ALPHA
        # [수정] BH 조건 분기 없이 동적 alpha를 계산해 라인 색상 지정
        link_col.append(hex_to_rgba(pat_color(i), round(alpha, 3)))

    gene_in_total = {}
    for (i, gname), z in pat_gene_z.items():
        gene_in_total[gname] = gene_in_total.get(gname, 0) + z
    gene_out_raw = {}
    for ti, edges_by_gene in pathway_gene_edges.items():
        if ti not in path_pos:
            continue
        for gname, edges in edges_by_gene.items():
            gene_out_raw[gname] = gene_out_raw.get(gname, 0) + sum(abs(z) for z in edges.values())

    for ti, edges_by_gene in pathway_gene_edges.items():
        if ti not in path_pos:
            continue
        for gname, edges in edges_by_gene.items():
            raw = sum(abs(z) for z in edges.values())
            scale = gene_in_total[gname] / gene_out_raw[gname] if gene_out_raw[gname] else 1.0
            src.append(n_pat_d + gene_pos[gname])
            tgt.append(n_pat_d + n_gene_d + path_pos[ti])
            val.append(raw * scale)
            link_col.append(hex_to_rgba(PATH_COLORS[ti], 0.45))

    link_order = sorted(range(len(src)), key=lambda k: (tgt[k], src[k]))
    src = [src[k] for k in link_order]
    tgt = [tgt[k] for k in link_order]
    val = [val[k] for k in link_order]
    link_col = [link_col[k] for k in link_order]

    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(label=node_labels, color=node_colors, pad=5, thickness=12, line=dict(width=0)),
        link=dict(source=src, target=tgt, value=val, color=link_col),
    ))
    
    fig.update_layout(width=1200, height=max(700, (n_pat_d + n_gene_d) * 13), font_size=11,
                      title=f'{disp_name}: patient -> leading-edge gene -> pathway'
                            f'<br><sup>Link opacity dynamically scaled by |Z| score (|Z| range: {z_min:.2f} ~ {z_max:.2f})</sup>')

    svg = fig.to_image(format='svg').decode('utf-8')
    svg = re.sub(r'text-shadow:[^;"]*;', '', svg)
    out_path = FIGDIR / f'{stem}_sankey.png'
    cairosvg.svg2png(bytestring=svg.encode('utf-8'), write_to=str(out_path), scale=2)
    print(f'{disp_name}: {n_pat_d}/{n_pat_total} patients shown, {n_gene_d} genes, '
          f'{len(used_ti)}/{len(terms)} pathways -> {out_path}')
    return fig

In [ ]:
for stem in SLUG_MAP:
    fig = build_sankey(stem)
    if fig is not None:
        fig.show()

/tmp/ipykernel_5190/956463897.py:169: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  svg = fig.to_image(format='svg').decode('utf-8')


Tuberculosis: 101/101 patients shown, 98 genes, 12/12 pathways -> SignalTrendanlaysis/Figures/Tuberculosis_sankey.png


/tmp/ipykernel_5190/956463897.py:169: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  svg = fig.to_image(format='svg').decode('utf-8')


Pancreatitis: 69/79 patients shown, 65 genes, 7/7 pathways -> SignalTrendanlaysis/Figures/Pancreatitis_sankey.png


In [8]:
import pickle

import numpy as np
import pandas as pd

import MixedEffectsModeling.config as config
from MixedEffectsModeling.PerSamplePathwayAnalysis.pathway_convergence import gene_sig_at_q
from MixedEffectsModeling.SignalTrendAnalysis.run_leading_edge_pattern import CURDIR, SLUG_MAP, parse_curation

PCDIR = config.PATHWAY_CONV_DIR
OUT_PATH = CURDIR / "sankey5_edges.csv"

# collapses SUMMARY.md's finer category labels (inflammatory/infectious, inflammatory/vascular)
# into the two-way cancer/inflammatory split the flagship comparison uses
CATEGORY = {
    "Tuberculosis": "inflammatory", "Pancreatitis": "inflammatory", "Pre-eclampsia": "inflammatory",
    "Pancreatic_Cancer": "cancer", "Colorectal_Cancer": "cancer", "Lung_Cancer": "cancer",
    "Esophagus_Cancer": "cancer", "Stomach_Cancer": "cancer",
    "Liver_Cancer_Roskams-Hieter": "cancer", "Liver_Cancer_Chen": "cancer",
}

# SUMMARY.md "Oncogenesis-specificity of the recurring cancer terms": Roskams-Hieter's upstream
# "Signaling By Hippo" and Chen/Pancreatic's downstream "YAP1/WWTR1-stimulated Gene Expression"
# are the same tumor-suppressor-inactivation axis under different GSEA Term strings, so exact-Term
# matching alone misses the 3rd cohort. Merge only these three (stem, term) hits into one axis
# label -- explicitly NOT Pre-eclampsia's own "Signaling By Hippo" hit, which is the opposite NES
# direction and a different, unrelated biology (trophoblast invasion, not oncogenesis); merging it
# in would conflate a real 3-cohort oncogenic reproduction with a cross-category coincidence.
HIPPO_YAP_AXIS = "Hippo-inactivation / YAP1-TAZ axis (oncogenic)"
TERM_GROUP_OVERRIDE = {
    ("Liver_Cancer_Roskams-Hieter", "Signaling By Hippo R-HSA-2028269"): HIPPO_YAP_AXIS,
    ("Liver_Cancer_Chen", "YAP1- And WWTR1 (TAZ)-stimulated Gene Expression R-HSA-2032785"): HIPPO_YAP_AXIS,
    ("Pancreatic_Cancer", "YAP1- And WWTR1 (TAZ)-stimulated Gene Expression R-HSA-2032785"): HIPPO_YAP_AXIS,
}


def term_group(stem, term):
    return TERM_GROUP_OVERRIDE.get((stem, term), term)


def recurring_groups(stem_terms):
    group_stems = {}
    for stem, terms in stem_terms.items():
        for t in terms:
            group_stems.setdefault(term_group(stem, t), set()).add(stem)
    return {g: stems for g, stems in group_stems.items() if len(stems) >= 2}


if __name__ == "__main__":
    heat_data = pickle.load(open(CURDIR / "leading_edge_pattern_data.pkl", "rb"))

    stem_terms = {stem: parse_curation(CURDIR / f"{stem}.md")[1] for stem in SLUG_MAP}
    recur = recurring_groups(stem_terms)
    print(f"{len(recur)} recurring pathway groups across >=2 phenotypes:")
    for g, stems in recur.items():
        print(f"  {g}: {sorted(stems)}")

    rows = []
    for stem, slug in SLUG_MAP.items():
        pdir = PCDIR / slug
        d = pickle.load(open(pdir / "sig.pkl", "rb"))
        names_c = d["names_c"]
        sym2idx = {s: i for i, s in enumerate(d["universe_syms"])}
        Zu, Fm = pickle.load(open(pdir / "universe.pkl", "rb"))
        bh_sig = gene_sig_at_q(Zu, Fm, config.PATHWAY_CONV_PARAMS["fdr_q"])

        for term in stem_terms[stem]:
            group = term_group(stem, term)
            if group not in recur:
                continue
            key = (stem, term)
            if key not in heat_data:
                continue
            Zsub, genes = heat_data[key]
            hit = np.abs(Zsub) > config.PATHWAY_CONV_PARAMS["z_thresh"]
            ti = d["terms"].index(term)
            path_sig_col = d["path_sig"][:, ti]
            for i, sample in enumerate(names_c):
                for gc, gname in enumerate(genes):
                    if not hit[i, gc]:
                        continue
                    uc = sym2idx.get(gname)
                    rows.append(dict(
                        category=CATEGORY[stem], phenotype=stem, term=term, term_group=group, gene=gname,
                        sample=sample, z=float(Zsub[i, gc]),
                        bh_sig=bool(uc is not None and bh_sig[i, uc]),
                        path_sig=bool(path_sig_col[i]),
                    ))

    edges = pd.DataFrame(rows)
    edges.to_csv(OUT_PATH, index=False)
    print(f"{len(edges)} sample-gene-pathway edges, {edges['phenotype'].nunique()} phenotypes, "
          f"{edges['term_group'].nunique()} pathway groups ({edges['term'].nunique()} raw terms) -> {OUT_PATH}")


11 recurring pathway groups across >=2 phenotypes:
  Interferon Alpha/Beta Signaling R-HSA-909733: ['Pancreatitis', 'Tuberculosis']
  Interferon Gamma Signaling R-HSA-877300: ['Pancreatitis', 'Tuberculosis']
  Neutrophil Degranulation R-HSA-6798695: ['Pancreatitis', 'Tuberculosis']
  Complement and coagulation cascades: ['Liver_Cancer_Chen', 'Pancreatic_Cancer']
  Hippo-inactivation / YAP1-TAZ axis (oncogenic): ['Liver_Cancer_Chen', 'Liver_Cancer_Roskams-Hieter', 'Pancreatic_Cancer']
  Mismatch repair: ['Colorectal_Cancer', 'Esophagus_Cancer', 'Lung_Cancer', 'Stomach_Cancer']
  Glycolysis / Gluconeogenesis: ['Colorectal_Cancer', 'Lung_Cancer']
  MAP3K8 (TPL2)-dependent MAPK1/3 Activation R-HSA-5684264: ['Colorectal_Cancer', 'Esophagus_Cancer', 'Stomach_Cancer']
  Telomere Extension By Telomerase R-HSA-171319: ['Esophagus_Cancer', 'Stomach_Cancer']
  Cell Cycle R-HSA-1640170: ['Esophagus_Cancer', 'Stomach_Cancer']
  Binding And Uptake Of Ligands By Scavenger Receptors R-HSA-2173782: ['L